# Stage 2: Few-Shot Meta-Learning for Contract Renegotiation

This notebook consolidates the Stage 2 meta-learning workflow into a single orchestration and evaluation surface. Stage 1 has already produced weak supervision labels and a pretrained TabularMLP. Stage 2 evaluates whether that pretrained representation can be adapted to the held-out target department, Logistics, using scarce gold labels.

The notebook intentionally avoids implementing PyTorch inner-loop or outer-loop mathematics directly. Those procedures are maintained in the source package under `src/master_thesis/`. This notebook therefore focuses on experiment configuration, pipeline execution, artifact collection, and thesis-facing visual evaluation.

## 1. Environment Setup and Warning Suppression

The Stage 2 pipeline combines local project modules, PyTorch models, YAML configuration files, and saved model artifacts. This setup cell establishes the project import path, suppresses non-critical warnings, and imports only the orchestration and plotting tools needed by the notebook. Suppressing `UserWarning` and `FutureWarning` keeps the notebook readable while preserving real exceptions that indicate broken logic.

In [1]:
from __future__ import annotations

import copy
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from master_thesis.stage2 import (
    load_stage2_config,
    resolve_stage2_paths,
    run_stage2_pipeline,
)

from master_thesis.plotting import plot_adaptation_curve

## 2. Mathematical Theory and Motivation

Stage 2 investigates whether a weakly supervised pretrained neural network can be specialized to a new procurement department using only a small number of gold labels. Standard fine-tuning updates model parameters directly on the few available support examples. In a high-dimensional tabular neural network, this can lead to overfitting because the support set is too small to constrain the full parameter space.

Meta-learning methods address this problem by changing the learning objective. Rather than only learning parameters that perform well globally, MAML and FOMAML seek an initialization that can be adapted efficiently after a few gradient steps. ANIL uses a related idea but restricts inner-loop adaptation to the final prediction head, making it more conservative and often more stable in very low-data regimes.

The empirical question in this notebook is therefore not simply whether the model predicts renegotiation. The question is whether meta-learning produces a more reliable adaptation process than standard fine-tuning when Logistics is treated as a held-out target department.

## Thesis Figure Alignment for Stage 2

This notebook supports the Stage 2 figures defined in the thesis figure plan. The purpose of these plots is not only to show which method performs best, but to make the few-shot adaptation argument academically transparent.

The Stage 2 plots should answer five questions:

1. **Does adaptation improve over the weak-pretrained baseline?**  
   Addressed by the method comparison plot across Fine-Tuning, ANIL, FOMAML, and MAML.

2. **Are the results stable across repeated support/query episodes?**  
   Addressed by repeated-episode distribution plots.

3. **How sensitive are the methods to the number of gold labels?**  
   Addressed by K-shot performance curves.

4. **How sensitive are the methods to adaptation depth?**  
   Addressed by inner-step performance curves and adaptation-loss curves.

5. **Are the predicted probabilities reliable for decision support?**  
   Addressed by log-loss, ECE, calibration-focused comparisons, and prediction-probability diagnostics if per-contract scores are available.

For the thesis, this notebook primarily supports Figure 18 to Figure 22 from the figure plan: Stage 2 method comparison, repeated-episode distribution, K-shot scaling, inner-step scaling, and prediction-probability diagnostics.


## 3. Stage 2 Configuration Definition

The following cell defines the few-shot experimental setup. The values are intentionally centralized so that support-set size, adaptation depth, and learning rates can be changed without editing downstream code. This is important for a thesis pipeline because the same notebook should support both a fast smoke test and a more complete benchmark run.

The default setup below evaluates 5-shot positive and 5-shot negative support sets for each Logistics episode. The benchmark methods are fine-tuning, ANIL, FOMAML, and full MAML.

In [2]:
CONFIG_PATH = PROJECT_ROOT / "experiments" / "stage2_config.yaml"

METHODS = ["finetune", "anil", "fomaml", "maml"]

FEW_SHOT_SETUP = {
    "n_support_pos": 5,
    "n_support_neg": 5,
    "inner_steps": 5,
    "inner_lr": 1e-3,
    "outer_lr": 1e-3,
    "meta_iterations": 100,
    "n_repeats": 20,
}

config = load_stage2_config(CONFIG_PATH)
paths = resolve_stage2_paths(config)

print("Loaded Stage 2 configuration from:", CONFIG_PATH)
print("Stage 1 model path:", paths.stage1_model_path)
print("Stage 1 preprocessor path:", paths.stage1_preprocessor_path)
print("Few-shot setup:", FEW_SHOT_SETUP)

Loaded Stage 2 configuration from: /Users/Thomas/Desktop/Master Thesis/experiments/stage2_config.yaml
Stage 1 model path: /Users/Thomas/Desktop/Master Thesis/models/stage_1/A_weak_only/mlp_pretrained.pt
Stage 1 preprocessor path: /Users/Thomas/Desktop/Master Thesis/models/stage_1/A_weak_only/mlp_pretrained_preprocessor.joblib
Few-shot setup: {'n_support_pos': 5, 'n_support_neg': 5, 'inner_steps': 5, 'inner_lr': 0.001, 'outer_lr': 0.001, 'meta_iterations': 100, 'n_repeats': 20}


## 4. Apply Notebook-Level Configuration Overrides

This cell applies the notebook-level few-shot settings to a copy of the YAML configuration. The original YAML file remains the source of truth, while the notebook can still override selected values for controlled experiments. This separation makes the experiment reproducible and avoids hidden changes in the source configuration file.

In [3]:
def make_method_config(base_config: dict, method: str, overrides: dict) -> dict:
    method_config = copy.deepcopy(base_config)
    method_config["method"] = method

    method_config.setdefault("support_config", {})
    method_config["support_config"]["n_support_pos"] = overrides["n_support_pos"]
    method_config["support_config"]["n_support_neg"] = overrides["n_support_neg"]

    method_config.setdefault("meta_config", {})
    method_config["meta_config"]["inner_steps"] = overrides["inner_steps"]
    method_config["meta_config"]["inner_lr"] = overrides["inner_lr"]
    method_config["meta_config"]["outer_lr"] = overrides["outer_lr"]
    method_config["meta_config"]["meta_iterations"] = overrides["meta_iterations"]
    method_config["meta_config"]["n_repeats"] = overrides["n_repeats"]

    return method_config

benchmark_configs = {
    method: make_method_config(config, method, FEW_SHOT_SETUP)
    for method in METHODS
}

print("Prepared method configurations:", list(benchmark_configs))

Prepared method configurations: ['finetune', 'anil', 'fomaml', 'maml']


## 5. Execute the Meta-Learning Pipeline

The heavy learning logic is delegated to `run_stage2_pipeline`. This function is expected to prepare the department tasks, load the Stage 1 pretrained model and preprocessor, run the selected adaptation method, evaluate the target Logistics episodes, and save method-specific artifacts.

Keeping the execution cell compact is deliberate. The notebook should document and orchestrate experiments, while the reusable implementation lives in the project package where it can be tested, versioned, and reused from scripts.

In [4]:
stage2_results = {}

for method, method_config in benchmark_configs.items():
    print(f"Running Stage 2 method: {method}")
    stage2_results[method] = run_stage2_pipeline(method_config)

print("Completed Stage 2 benchmark methods:", list(stage2_results))

Running Stage 2 method: finetune


/Users/Thomas/Desktop/Master Thesis/src/master_thesis/stage2.py:252: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["lpi_relative_risk"] = df["lpi_below_supplier_median"]


Stage 2 task table created
Rows: 186
Unique contracts: 39
Departments: 9
Positive labels: 100
Negative labels: 86
Department task summary created
Departments: 9
Total labeled rows: 186
Total positive rows: 100
Total negative rows: 86
Total labeled contracts: 39
Total positive contracts: 20
Total negative contracts: 19
Department task summary created
Departments: 9
Total labeled rows: 186
Total positive rows: 100
Total negative rows: 86
Total labeled contracts: 39
Total positive contracts: 20
Total negative contracts: 19
Department validity filtering completed
Valid departments: 1
Invalid departments: 8
Running Stage 2 method: anil
Stage 2 task table created
Rows: 186
Unique contracts: 39
Departments: 9
Positive labels: 100
Negative labels: 86
Department task summary created
Departments: 9
Total labeled rows: 186
Total positive rows: 100
Total negative rows: 86
Total labeled contracts: 39
Total positive contracts: 20
Total negative contracts: 19
Department task summary created
Departmen

/Users/Thomas/Desktop/Master Thesis/src/master_thesis/stage2.py:252: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["lpi_relative_risk"] = df["lpi_below_supplier_median"]


ANIL Meta-Iteration 010 | Global Loss: 0.0000 | Skipped: 0
ANIL Meta-Iteration 020 | Global Loss: 0.0000 | Skipped: 0
ANIL Meta-Iteration 030 | Global Loss: 8.6589 | Skipped: 0
ANIL Meta-Iteration 040 | Global Loss: 3.3026 | Skipped: 0
ANIL Meta-Iteration 050 | Global Loss: 0.0000 | Skipped: 0
ANIL Meta-Iteration 060 | Global Loss: 0.0000 | Skipped: 0
ANIL Meta-Iteration 070 | Global Loss: 0.0000 | Skipped: 0
ANIL Meta-Iteration 080 | Global Loss: 0.0000 | Skipped: 0
ANIL Meta-Iteration 090 | Global Loss: 0.0000 | Skipped: 0
ANIL Meta-Iteration 100 | Global Loss: 0.0000 | Skipped: 0
Running Stage 2 method: fomaml
Stage 2 task table created
Rows: 186
Unique contracts: 39
Departments: 9
Positive labels: 100
Negative labels: 86
Department task summary created
Departments: 9
Total labeled rows: 186
Total positive rows: 100
Total negative rows: 86
Total labeled contracts: 39
Total positive contracts: 20
Total negative contracts: 19
Department task summary created
Departments: 9
Total labele

/Users/Thomas/Desktop/Master Thesis/src/master_thesis/stage2.py:252: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["lpi_relative_risk"] = df["lpi_below_supplier_median"]


FOMAML Meta-Iteration 010 | Global Loss: 5678720512.0000 | Skipped: 0
FOMAML Meta-Iteration 020 | Global Loss: 0.0000 | Skipped: 0
FOMAML Meta-Iteration 030 | Global Loss: 0.0000 | Skipped: 0
FOMAML Meta-Iteration 040 | Global Loss: 0.0000 | Skipped: 0
FOMAML Meta-Iteration 050 | Global Loss: 0.0000 | Skipped: 0
FOMAML Meta-Iteration 060 | Global Loss: 0.0000 | Skipped: 0
FOMAML Meta-Iteration 070 | Global Loss: 0.0000 | Skipped: 0
FOMAML Meta-Iteration 080 | Global Loss: 0.0000 | Skipped: 0
FOMAML Meta-Iteration 090 | Global Loss: 0.0000 | Skipped: 0
FOMAML Meta-Iteration 100 | Global Loss: 0.0000 | Skipped: 0
Running Stage 2 method: maml
Stage 2 task table created
Rows: 186
Unique contracts: 39
Departments: 9
Positive labels: 100
Negative labels: 86
Department task summary created
Departments: 9
Total labeled rows: 186
Total positive rows: 100
Total negative rows: 86
Total labeled contracts: 39
Total positive contracts: 20
Total negative contracts: 19
Department task summary created


/Users/Thomas/Desktop/Master Thesis/src/master_thesis/stage2.py:252: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["lpi_relative_risk"] = df["lpi_below_supplier_median"]


MAML Meta-Iteration 010 | Global Loss: 0.0778 | Skipped: 0
MAML Meta-Iteration 020 | Global Loss: 0.0008 | Skipped: 0
MAML Meta-Iteration 030 | Global Loss: 0.0002 | Skipped: 0
MAML Meta-Iteration 040 | Global Loss: 0.0001 | Skipped: 0
MAML Meta-Iteration 050 | Global Loss: 0.0001 | Skipped: 0
MAML Meta-Iteration 060 | Global Loss: 0.0001 | Skipped: 0
MAML Meta-Iteration 070 | Global Loss: 0.0000 | Skipped: 0
MAML Meta-Iteration 080 | Global Loss: 0.0000 | Skipped: 0
MAML Meta-Iteration 090 | Global Loss: 0.0000 | Skipped: 0
MAML Meta-Iteration 100 | Global Loss: 0.0000 | Skipped: 0
Completed Stage 2 benchmark methods: ['finetune', 'anil', 'fomaml', 'maml']


## 6. Extract Raw Episode Metrics

The pipeline returns raw repeated-episode metrics for each method. These raw metrics are the foundation for the benchmark plots because they preserve variation across Logistics support/query splits. Reporting repeated-episode metrics is more rigorous than reporting a single split, because few-shot results can be sensitive to which examples are sampled into the support set.

In [6]:
def _metrics_to_frame(metrics) -> pd.DataFrame:
    if metrics is None:
        return pd.DataFrame()

    if isinstance(metrics, pd.DataFrame):
        return metrics.copy()

    if isinstance(metrics, list):
        rows = [row for row in metrics if isinstance(row, dict) and row]
        return pd.DataFrame(rows)

    if isinstance(metrics, dict):
        if "raw_metrics" in metrics:
            return _metrics_to_frame(metrics["raw_metrics"])
        if "metrics" in metrics:
            return _metrics_to_frame(metrics["metrics"])

        scalar_metrics = {
            k: v for k, v in metrics.items()
            if isinstance(v, (int, float, str, bool)) or v is None
        }
        return pd.DataFrame([scalar_metrics]) if scalar_metrics else pd.DataFrame()

    return pd.DataFrame()


def extract_stage2_metrics(stage2_results: dict) -> pd.DataFrame:
    frames = []
    diagnostics = []

    for method, result in stage2_results.items():
        payload = result.get("result", result) if isinstance(result, dict) else {}
        eval_result = payload.get("target_eval_result", {}) if isinstance(payload, dict) else {}

        raw_df = _metrics_to_frame(eval_result.get("raw_metrics"))
        agg_df = _metrics_to_frame(eval_result.get("metrics"))

        if not raw_df.empty:
            df_method = raw_df.copy()
            df_method["method"] = method
            df_method["metric_source"] = "raw_metrics"
            frames.append(df_method)
            continue

        if not agg_df.empty:
            df_method = agg_df.copy()
            df_method["method"] = method
            df_method["metric_source"] = "metrics"
            frames.append(df_method)
            continue

        diagnostics.append({
            "method": method,
            "status": payload.get("status", "unknown") if isinstance(payload, dict) else "not_dict",
            "target_department": payload.get("target_department", "unknown") if isinstance(payload, dict) else "unknown",
            "payload_keys": list(payload.keys()) if isinstance(payload, dict) else [],
            "target_eval_keys": list(eval_result.keys()) if isinstance(eval_result, dict) else [],
            "reason": "No usable raw_metrics or metrics returned.",
        })

    if not frames:
        df_diagnostics = pd.DataFrame(diagnostics)
        display(df_diagnostics)
        raise ValueError(
            "No usable Stage 2 metrics were found. "
            "Most likely the target department could not form valid support/query episodes "
            "from the current gold labels."
        )

    if diagnostics:
        print("Some methods did not return usable metrics:")
        display(pd.DataFrame(diagnostics))

    return pd.concat(frames, ignore_index=True)


df_stage2_metrics = extract_stage2_metrics(stage2_results)

df_stage2_metrics["method"] = (
    df_stage2_metrics["method"]
    .astype(str)
    .str.replace("fine_tune", "finetune", regex=False)
    .str.lower()
)

display(df_stage2_metrics.head())
print("Metric table shape:", df_stage2_metrics.shape)
print("Metric sources:")
display(df_stage2_metrics[["method", "metric_source"]].drop_duplicates())

,method,status,target_department,payload_keys,target_eval_keys,reason
0,finetune,no_target_episodes_available,Logistics,"[method, status, target_department]",[],No usable raw_metrics or metrics returned.
1,anil,fully_integrated,Logistics,"[method, status, target_department, weight_pat...","[method, status, metrics]",No usable raw_metrics or metrics returned.
2,fomaml,fully_integrated,Logistics,"[method, status, target_department, weight_pat...","[method, status, metrics]",No usable raw_metrics or metrics returned.
3,maml,fully_integrated,Logistics,"[method, status, target_department, weight_pat...","[method, status, metrics]",No usable raw_metrics or metrics returned.


ValueError: No usable Stage 2 metrics were found. Most likely the target department could not form valid support/query episodes from the current gold labels.

## 7. Aggregate Benchmark Summary

The following table summarizes mean and standard deviation for the main Stage 2 metrics. AUROC measures discrimination, log-loss measures probability quality, ECE measures calibration, and NDCG@10 measures ranking quality for top-K contract prioritization. Together, these metrics reflect both statistical performance and practical decision-support value.

In [ ]:
MAIN_METRICS = ["gold_auroc", "gold_logloss", "gold_ece", "ndcg_at_10"]

summary_frames = []
for metric in MAIN_METRICS:
    if metric in df_stage2_metrics.columns:
        metric_summary = (
            df_stage2_metrics
            .groupby("method")[metric]
            .agg(["mean", "std"])
            .rename(columns={"mean": f"{metric}_mean", "std": f"{metric}_std"})
        )
        summary_frames.append(metric_summary)

df_stage2_summary = pd.concat(summary_frames, axis=1).reset_index()
display(df_stage2_summary)

## Stage 2 Plot Coverage Checklist

The following sections should be interpreted as the Stage 2 visual evidence package.

| Thesis figure | Notebook section | Purpose |
|---|---|---|
| Figure 18 — Stage 2 method comparison | Main benchmark visualization | Compares methods across discrimination, ranking, and probability quality |
| Figure 19 — Repeated episode distribution | Repeated-episode AUROC distribution | Shows whether few-shot results are robust or split-dependent |
| Figure 20 — K-shot performance curve | K-shot scaling | Shows how performance changes as more gold labels become available |
| Figure 21 — Inner-step adaptation curve | Inner-step scaling and adaptation loss | Shows whether more adaptation helps or causes overfitting |
| Figure 22 — Prediction probability distribution | Prediction probability diagnostic | Detects overconfidence, threshold collapse, or probability compression |

If a plot is skipped because the necessary artifact is missing, this should be reported as a pipeline limitation rather than hidden. For example, prediction-probability distributions require per-contract prediction outputs, while repeated-episode boxplots require raw episode-level metrics.


## 8. Main Benchmark Visualization: AUROC, Log-Loss, and NDCG

This figure provides the headline comparison between adaptation methods. A strong Stage 2 method should improve discrimination, reduce log-loss, and preserve strong ranking performance. Plotting these metrics side by side makes it possible to distinguish a method that ranks contracts well from one that also produces reliable probability estimates.

In [ ]:
plot_metrics = [
    ("gold_auroc", "AUROC", True),
    ("gold_logloss", "Log-Loss", False),
    ("ndcg_at_10", "NDCG@10", True),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (metric, title, higher_is_better) in zip(axes, plot_metrics):
    plot_df = (
        df_stage2_metrics
        .groupby("method")[metric]
        .agg(["mean", "std"])
        .reindex(METHODS)
        .reset_index()
    )

    ax.bar(plot_df["method"], plot_df["mean"], yerr=plot_df["std"], capsize=4)
    ax.set_title(title)
    ax.set_xlabel("Method")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=25)

fig.suptitle("Stage 2 benchmark on Logistics repeated episodes", y=1.05)
plt.tight_layout()
plt.show()

## 9. Calibration-Focused Benchmark

Calibration is important because the proposed model is intended for decision support rather than only hard classification. A method that achieves good ranking but produces overconfident probabilities may still be problematic for procurement stakeholders. This plot therefore isolates ECE and log-loss, both of which penalize unreliable probability estimates.

In [ ]:
calibration_metrics = ["gold_logloss", "gold_ece"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, metric in zip(axes, calibration_metrics):
    plot_df = (
        df_stage2_metrics
        .groupby("method")[metric]
        .mean()
        .reindex(METHODS)
        .reset_index()
    )

    ax.bar(plot_df["method"], plot_df[metric])
    ax.set_title(metric)
    ax.set_xlabel("Method")
    ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

## 10. Repeated-Episode Distribution

Few-shot evaluation is inherently unstable because support/query splits are small. A boxplot over repeated Logistics episodes shows whether the observed ranking of methods is systematic or driven by a small number of favorable splits. This figure is therefore important for academic credibility.

In [ ]:
metric = "gold_auroc"

box_data = [
    df_stage2_metrics.loc[df_stage2_metrics["method"] == method, metric].dropna()
    for method in METHODS
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(box_data, tick_labels=METHODS)
ax.set_title("Repeated-episode AUROC distribution")
ax.set_xlabel("Method")
ax.set_ylabel("AUROC")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## How to Interpret the Stage 2 Plots in the Thesis

The Stage 2 figures should be written cautiously. They should not be used to claim that MAML is universally superior. Instead, they should be used to evaluate when meta-learning adds value beyond a strong weak-supervised pretrained MLP.

Recommended interpretation logic:

- If **Fine-Tuning performs best**, the thesis finding is that Stage 1 weak supervision learned a strong transferable representation, and simple adaptation may be sufficient under limited source-task diversity.
- If **FOMAML or MAML improves as k increases**, the thesis finding is that meta-learning becomes more useful once enough balanced gold labels are available.
- If **ANIL underperforms**, the thesis finding is that freezing the feature extractor is too restrictive, meaning department adaptation requires updates beyond the final classification head.
- If **log-loss or ECE is poor despite strong AUROC/NDCG**, the model may rank contracts well but produce unreliable probabilities. This matters because the business use case is decision support.
- If **episode variance is high**, repeated support/query sampling must be reported with mean ± standard deviation instead of relying on a single split.

The main evidence should always come from repeated metrics, K-shot curves, and calibration/ranking metrics. SHAP and embedding diagnostics are supporting interpretability tools, not standalone proof of model quality.


## 11. Few-Shot Adaptation Curves: Inner-Loop Step Scaling

A central claim of meta-learning is that useful adaptation should occur within a small number of gradient steps. This section visualizes how adaptation loss evolves over inner-loop steps for each method when such loss histories are available from the pipeline artifacts. The plotting utility `plot_adaptation_curve` is imported from `master_thesis.plotting` to keep the notebook aligned with the reusable project code.

In [ ]:
def extract_loss_histories(stage2_results: dict) -> dict[str, list[float]]:
    histories = {}

    for method, result in stage2_results.items():
        payload = result.get("result", result)
        history = payload.get("history", None)

        if history is None:
            continue

        df_history = pd.DataFrame(history)

        for candidate_col in ["support_loss", "inner_loss", "loss", "meta_loss"]:
            if candidate_col in df_history.columns:
                histories[method] = df_history[candidate_col].dropna().tolist()
                break

    return histories

loss_histories = extract_loss_histories(stage2_results)

if loss_histories:
    fig, ax = plot_adaptation_curve(
        loss_histories,
        title="Inner-loop adaptation loss by method",
    )
    plt.show()
else:
    print("No loss histories were found in the returned Stage 2 artifacts.")

## 12. Few-Shot K-Shot Scaling Configuration

The k-shot scaling experiment evaluates how performance changes as the amount of support supervision increases. This is one of the most important analyses for a few-shot thesis because it directly tests whether meta-learning is more sample-efficient than standard fine-tuning.

The cell below defines the support sizes to evaluate. Running the full sweep can be computationally expensive, so it is controlled by `RUN_KSHOT_SWEEP`. Set it to `True` when the benchmark should be executed.

In [ ]:
KSHOT_VALUES = [1, 2, 3, 5, 7]
KSHOT_METHODS = ["finetune", "anil", "fomaml", "maml"]

RUN_KSHOT_SWEEP = False

print("K-shot values:", KSHOT_VALUES)
print("Methods:", KSHOT_METHODS)
print("Run sweep:", RUN_KSHOT_SWEEP)

## 13. Execute K-Shot Scaling Sweep

This cell reuses `run_stage2_pipeline` with different support sizes. No PyTorch update logic is implemented inside the notebook; the notebook simply modifies the configuration and calls the same tested pipeline. The resulting metrics are stored in `df_kshot_metrics` for visualization.

In [ ]:
kshot_results = []

if RUN_KSHOT_SWEEP:
    for k in KSHOT_VALUES:
        for method in KSHOT_METHODS:
            k_config = make_method_config(config, method, FEW_SHOT_SETUP)
            k_config["support_config"]["n_support_pos"] = k
            k_config["support_config"]["n_support_neg"] = k

            print(f"Running {method} with {k}-shot support")
            result = run_stage2_pipeline(k_config)
            df_raw = extract_raw_metrics({method: result})
            df_raw["k_shot"] = k
            kshot_results.append(df_raw)

    df_kshot_metrics = pd.concat(kshot_results, ignore_index=True)
else:
    df_kshot_metrics = pd.DataFrame()
    print("K-shot sweep skipped. Set RUN_KSHOT_SWEEP = True to run it.")

## 14. Visualize K-Shot Performance Scaling

The k-shot curve shows whether performance improves smoothly as more gold labels become available. If MAML or FOMAML reaches strong performance with fewer support examples than fine-tuning, that provides direct evidence for the sample-efficiency motivation of Stage 2.

In [ ]:
if not df_kshot_metrics.empty:
    fig, ax = plt.subplots(figsize=(8, 5))

    for method in KSHOT_METHODS:
        method_df = (
            df_kshot_metrics[df_kshot_metrics["method"] == method]
            .groupby("k_shot")["gold_auroc"]
            .mean()
            .reset_index()
        )
        ax.plot(method_df["k_shot"], method_df["gold_auroc"], marker="o", label=method)

    ax.set_title("K-shot scaling on Logistics")
    ax.set_xlabel("Support examples per class")
    ax.set_ylabel("Mean AUROC")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No K-shot metrics available yet.")

## 15. Inner-Step Scaling Configuration

The inner-step scaling experiment evaluates how sensitive each adaptation method is to the number of gradient updates. This matters because real few-shot adaptation should be efficient. A method that requires many updates may be less useful operationally and more vulnerable to overfitting.

In [ ]:
INNER_STEP_VALUES = [0, 1, 3, 5, 10]
STEP_METHODS = ["finetune", "anil", "fomaml", "maml"]

RUN_STEP_SWEEP = False

print("Inner-step values:", INNER_STEP_VALUES)
print("Methods:", STEP_METHODS)
print("Run sweep:", RUN_STEP_SWEEP)

## 16. Execute Inner-Step Scaling Sweep

This cell evaluates the same methods across multiple adaptation depths. The purpose is to determine whether meta-learning methods improve quickly and whether standard fine-tuning becomes unstable as more gradient updates are applied.

In [ ]:
step_results = []

if RUN_STEP_SWEEP:
    for n_steps in INNER_STEP_VALUES:
        for method in STEP_METHODS:
            step_config = make_method_config(config, method, FEW_SHOT_SETUP)
            step_config["meta_config"]["inner_steps"] = n_steps

            print(f"Running {method} with {n_steps} inner steps")
            result = run_stage2_pipeline(step_config)
            df_raw = extract_raw_metrics({method: result})
            df_raw["inner_steps"] = n_steps
            step_results.append(df_raw)

    df_step_metrics = pd.concat(step_results, ignore_index=True)
else:
    df_step_metrics = pd.DataFrame()
    print("Inner-step sweep skipped. Set RUN_STEP_SWEEP = True to run it.")

## 17. Visualize Inner-Step Performance Scaling

This plot evaluates rapid adaptation in terms of predictive performance rather than only training loss. It complements the loss-based adaptation curve by showing whether additional inner-loop steps translate into improved query-set AUROC.

In [ ]:
if not df_step_metrics.empty:
    fig, ax = plt.subplots(figsize=(8, 5))

    for method in STEP_METHODS:
        method_df = (
            df_step_metrics[df_step_metrics["method"] == method]
            .groupby("inner_steps")["gold_auroc"]
            .mean()
            .reset_index()
        )
        ax.plot(method_df["inner_steps"], method_df["gold_auroc"], marker="o", label=method)

    ax.set_title("Inner-step scaling on Logistics")
    ax.set_xlabel("Inner-loop adaptation steps")
    ax.set_ylabel("Mean AUROC")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No inner-step metrics available yet.")

## 18. Save Consolidated Stage 2 Benchmark Tables

The final cell saves the aggregated benchmark outputs generated by the notebook. These CSV files provide a reproducible record of the repeated-episode evaluation and can be reused directly in the thesis results chapter.

In [ ]:
output_dir = PROJECT_ROOT / "reports" / "stage2"
output_dir.mkdir(parents=True, exist_ok=True)

metrics_path = output_dir / "stage2_repeated_episode_metrics.csv"
summary_path = output_dir / "stage2_method_summary.csv"

df_stage2_metrics.to_csv(metrics_path, index=False)
df_stage2_summary.to_csv(summary_path, index=False)

print("Saved repeated metrics to:", metrics_path)
print("Saved summary metrics to:", summary_path)

## Optional Thesis Extensions: Explainability and Representation Diagnostics

The thesis figure plan also recommends explainability and representation diagnostics after the Stage 2 benchmark:

- SHAP before vs after adaptation.
- View-level SHAP importance.
- Department/task clustering in the Stage 1 embedding space.
- Logistics gold-label separability before adaptation.
- Representation shift before and after adaptation.
- Embedding shift magnitude after adaptation.

These plots are useful if the necessary artifacts are exported from the model pipeline: per-contract predictions, SHAP values, feature names, and hidden-layer embeddings. They can help answer whether weak supervision already captures most department structure or whether few-shot adaptation meaningfully changes the learned representation. They should be treated as qualitative diagnostics and reported alongside quantitative metrics.
